In [1]:
import dask
from dask import delayed
from dask.distributed import Client
from dask_jobqueue import SLURMCluster
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import glob
import numpy as np

# Change working directory
import os
os.chdir(os.getcwd() + "/../")
print(os.getcwd())

import config
from uncertainty_experiments.uncertainty_nemi import compute_volumetric_nemi, compute_volume

C:\Users\yvjennig\PycharmProjects\github\ocean_clustering_and_validation


In [2]:
# Load all clustering runs
pack = []
for filename in tqdm(glob.glob(f"{config.output_dir_uncertainty}umap_dbscan_*.csv")):
    try:
        i = int(filename.split("_")[-1].rstrip(".csv"))

        # Load data
        df = pd.read_csv(filename)
        df.label = df.label + 1  # Make sure no label is -1 (noise in DBSCAN)
        pack.append([i, df])

    except ValueError:
        print(f"Skipping invalid file: {filename}")
        continue

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [00:12<00:00,  8.25it/s]

Skipping invalid file: output/uncertainty\umap_dbscan_uncertainty_metrics.csv


In [3]:
# Sort by size and compute volumes
for i, cl in tqdm(pack):
    clusters = cl.label
    n_clusters = len(clusters.unique())
    hist, _ = np.histogram(clusters, np.arange(n_clusters + 1))
    sorted_clusters = np.argsort(hist)[::-1]  # Sort from largest to smallest (if same size, last cluster is taken)
    new_labels = np.full(clusters.shape, np.nan)
    for new_label, old_label in enumerate(sorted_clusters):
        new_labels[clusters == old_label] = new_label
    cl["sorted_label"] = new_labels

    # Compute volume
    dlat, dlon = [1, 1]
    depths = np.append(np.sort(cl.LEV_M.unique()), 5000)
    cl.loc[:, "volume"] = cl.apply(compute_volume, axis=1, args=(dlat, dlon, depths))  # Careful with rounding

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [02:16<00:00,  1.37s/it]


In [4]:
# Define a function for processing each base_id
@dask.delayed
def process_base_id(df, base_id, pack, prefix):
    filename = f"{config.output_dir_uncertainty}{prefix}nemi_iteration{base_id}_uncertainty.csv"

    # Only do computations if the file does not exist
    if Path(filename).exists():
        return f"Skipping {filename}, already exists."

    # Compute NEMI labels and uncertainty
    final_labels, uncertainty = compute_volumetric_nemi(nemi_pack=pack, base_id=base_id)

    # Store results
    df["final_label"] = final_labels
    df["uncertainty"] = uncertainty * 100
    df = color_code_labels(df, column_name="final_label").rename({"color": "label_color"}, axis=1)
    
    # Save to CSV
    df.to_csv(filename, index=False)
    
    return f"Processed {filename}"

In [5]:
# Create the client for parallel execution
client = Client(n_workers=4, threads_per_worker=1)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 31.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:57750,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 4
Started: Just now,Total memory: 31.67 GiB
Comm: tcp://127.0.0.1:57770,Total threads: 1
Dashboard: http://127.0.0.1:57775/status,Memory: 7.92 GiB
Nanny: tcp://127.0.0.1:57753,


In [6]:
# Create a list of delayed tasks
prefix = "volume_"
df = pack[0][1]
tasks = [process_base_id(df=df, base_id=base_id, pack=pack, prefix=prefix) for base_id in range(config.n_iterations_uncertainty)]

In [9]:
# Run tasks in parallel
results = dask.compute(*tasks)

C:\Users\yvjennig\Anaconda3\lib\site-packages\distributed\client.py:3362: UserWarning: Sending large graph of size 636.11 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# Print results
for res in results:
    print(res)